# 🌾 Project 4: FPO Procurement & Supply Chain Diagnostic
## Phase 1: Python EDA, Data Cleaning & Quality Grading Pipeline

This notebook processes raw procurement receipts (`fpo_procurement_raw.csv`) from 15 village collection centers in Andhra Pradesh, cleans text formatting, handles missing values, calculates non-linear moisture and refraction penalties, assigns quality grades (Grade A, B, C), and exports clean datasets for SQL and Excel auditing.

In [1]:
import pandas as pd
import numpy as np

# 1. Load Raw Dataset
df_raw = pd.read_csv('fpo_procurement_raw.csv')
print(f"Raw Records Loaded: {len(df_raw)}")
df_raw.head()

## Task 1: Data Hygiene & Deduplication
We standardize string formatting across collection centers, farmer IDs, and crop names, and drop duplicate records.

In [2]:
# Drop duplicates
df_clean = df_raw.drop_duplicates(subset=['batch_id']).copy()

# Standardize text fields
df_clean['collection_center'] = df_clean['collection_center'].astype(str).str.strip().str.title()
df_clean['crop_name'] = df_clean['crop_name'].astype(str).str.strip()
df_clean['farmer_id'] = df_clean['farmer_id'].astype(str).str.strip()

print(f"Clean Records after Deduplication: {len(df_clean)}")

## Task 2: Quality Penalty & Net Weight Calculation
We compute total quality penalty percentages based on measured moisture content (%) and foreign matter refraction (%), and derive net payable weight in Quintals.

In [3]:
# Moisture Penalty Formula
# Base Moisture Target: Paddy 14%, Chilli 10%, Maize 13.5%, Turmeric 10.5%, Toor Dal 12%
df_clean['moisture_penalty_pct'] = np.maximum(0, df_clean['measured_moisture_pct'] - 14.0) * 0.8
df_clean['total_penalty_pct'] = df_clean['moisture_penalty_pct'] + df_clean['refraction_pct']

# Net Weight Calculation
df_clean['net_weight_quintals'] = df_clean['gross_weight_quintals'] * (1 - df_clean['total_penalty_pct'] / 100.0)
df_clean['net_weight_quintals'] = df_clean['net_weight_quintals'].round(2)

print(f"Total Gross Weight: {df_clean['gross_weight_quintals'].sum():,.2f} Qtl")
print(f"Total Net Weight:   {df_clean['net_weight_quintals'].sum():,.2f} Qtl")
print(f"Total Shrinkage:    {df_clean['gross_weight_quintals'].sum() - df_clean['net_weight_quintals'].sum():,.2f} Qtl")

## Task 3: Quality Grading & Price Realization
We assign Grade A (Score >= 88, 5% premium), Grade B (Score 80-87), and Grade C (Score < 80, 8% markdown), and compute gross farmer payable amount and buyer invoice values.

In [4]:
def assign_grade(score):
    if score >= 88:
        return 'Grade A'
    elif score >= 80:
        return 'Grade B'
    else:
        return 'Grade C'

df_clean['quality_grade'] = df_clean['quality_score'].apply(assign_grade)

# Realized Price
df_clean['realized_price_per_qtl'] = df_clean['base_msp_per_quintal'] * np.where(
    df_clean['quality_grade'] == 'Grade A', 1.05,
    np.where(df_clean['quality_grade'] == 'Grade C', 0.92, 1.00)
)

# Financial Metrics
df_clean['gross_farmer_payable_inr'] = (df_clean['net_weight_quintals'] * df_clean['base_msp_per_quintal']).round(2)
df_clean['logistics_cost_inr'] = (df_clean['gross_weight_quintals'] * df_clean['transport_rate_per_qtl']).round(2)
df_clean['buyer_invoice_value_inr'] = (df_clean['net_weight_quintals'] * df_clean['realized_price_per_qtl']).round(2)
df_clean['fpo_handling_fee_inr'] = (df_clean['net_weight_quintals'] * 35.0).round(2)
df_clean['net_fpo_margin_inr'] = (df_clean['buyer_invoice_value_inr'] - df_clean['gross_farmer_payable_inr'] - df_clean['logistics_cost_inr']).round(2)

print('Grade Counts:')
print(df_clean['quality_grade'].value_counts())
print(f'\nTotal Farmer Payable: INR {df_clean["gross_farmer_payable_inr"].sum():,.2f}')
print(f'Total Logistics Cost:  INR {df_clean["logistics_cost_inr"].sum():,.2f}')
print(f'Total Buyer Invoice:   INR {df_clean["buyer_invoice_value_inr"].sum():,.2f}')
print(f'Total Net FPO Margin:  INR {df_clean["net_fpo_margin_inr"].sum():,.2f}')

## Task 4: Collection Center Summary & Export
We aggregate performance metrics across the 15 village collection centers and export clean CSVs.

In [5]:
summary = df_clean.groupby('collection_center').agg(
    total_batches=('batch_id', 'count'),
    total_gross_quintals=('gross_weight_quintals', 'sum'),
    total_net_quintals=('net_weight_quintals', 'sum'),
    avg_moisture_penalty_pct=('moisture_penalty_pct', 'mean'),
    avg_quality_score=('quality_score', 'mean'),
    total_farmer_payout_inr=('gross_farmer_payable_inr', 'sum'),
    total_logistics_cost_inr=('logistics_cost_inr', 'sum'),
    total_buyer_invoice_inr=('buyer_invoice_value_inr', 'sum'),
    total_fpo_margin_inr=('net_fpo_margin_inr', 'sum')
).reset_index()

# Round numbers
summary = summary.round(2)

# Export files
df_clean.to_csv('fpo_procurement_clean.csv', index=False)
summary.to_csv('collection_center_summary.csv', index=False)
print('Successfully exported fpo_procurement_clean.csv and collection_center_summary.csv!')